## Purpose:
Train and evaluate an XGBoost classifier to predict in-hospital mortality for ICU patients using SQL-engineered features from the MIMIC-III demo dataset. This notebook serves as the  proof-of-concept for Track 1 from raw database query to a trained, explainable ML model.

## Context:
The MIMIC-III demo contains ~100 ICU stays with a high mortality rate relative to the full dataset, creating a significant class imbalance. Standard accuracy metrics are misleading under these conditions, so this notebook uses AUROC as the primary evaluation metric and applies scale_pos_weight in XGBoost to compensate for the skewed class distribution.
All features are pulled directly from the SQL views built in sql/features.sql via tracks/sql_features.py — no feature engineering happens inside this notebook. The notebook's job is purely modeling: load features, handle imbalance, train, evaluate, and explain.
This notebook is intentionally scoped to the demo dataset as a pipeline prototype. Results should be interpreted as proof-of-concept only — production-quality performance requires the full MIMIC-III dataset (~46,000 ICU stays).

# Section 1 Imports

In [1]:
# Data handling
import pandas as pd
import numpy as np 

# Database
import sqlite3

# Model 
from xgboost import XGBClassifier

# Class imbalance 
from imblearn.over_sampling import SMOTE

# Sklearn utilities 
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, RocCurveDisplay
from sklearn.preprocessing import StandardScaler

# Explainability 
import shap

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Extras 
import warnings
from pathlib import Path

c:\Users\delga\anaconda3\envs\clinical-intelligence\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Section 2 Configuration 

In [2]:
# -- Setup path to mimic.db --
DB_PATH = Path("../db/mimic.db")

# -- Define target column --
TARGET = "died_in_hospital"

# -- Setup Random seeds for Reproducibility --
SEED = 42 
np.random.seed(SEED)

# -- Class Imbalance Placeholder --
SCALE_POS_WEIGHT = None 

# -- Supress any warnings --
warnings.filterwarnings("ignore")

# -- Plot Style -- 
sns.set_style("whitegrid")

# -- Sanity Check -- 
assert DB_PATH.exists(), f"Database not found at {DB_PATH}"
print(f"✅ Database found: {DB_PATH}")

✅ Database found: ..\db\mimic.db


# Section 3 Loading Features from the Database

In [4]:
# Connect to Database 
conn = sqlite3.connect(DB_PATH)

# Execute SQL file to create views in mimic.db
with open("../sql/features.sql") as f:
    sql = f.read()
conn.executescript(sql)

# Query the features from the databse 
v_features = pd.read_sql("SELECT * FROM v_features", conn)

# Query v_features view into a DataFrame
df = pd.read_sql("SELECT * FROM v_features", conn)

# Close connection
conn.close()

# Sanity checks
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (136, 89)
Columns: ['subject_id', 'hadm_id', 'icustay_id', 'gender', 'admission_type', 'insurance', 'ethnicity', 'dob', 'dod', 'intime', 'outtime', 'admittime', 'dischtime', 'icu_los_hours', 'hosp_los_days', 'age_at_admit', 'died_in_hospital', 'hr_mean_24h', 'hr_min_24h', 'hr_max_24h', 'sbp_mean_24h', 'sbp_min_24h', 'sbp_max_24h', 'spo2_mean_24h', 'spo2_min_24h', 'spo2_max_24h', 'resp_mean_24h', 'resp_min_24h', 'resp_max_24h', 'temp_mean_24h', 'temp_min_24h', 'temp_max_24h', 'gcs_mean_24h', 'gcs_min_24h', 'gcs_max_24h', 'n_vital_measurements_24h', 'hematocrit_mean_24h', 'hematocrit_min_24h', 'potassium_mean_24h', 'potassium_min_24h', 'potassium_max_24h', 'sodium_mean_24h', 'sodium_min_24h', 'sodium_max_24h', 'creatinine_mean_24h', 'creatinine_max_24h', 'chloride_mean_24h', 'bun_mean_24h', 'bun_max_24h', 'bicarb_mean_24h', 'bicarb_min_24h', 'anion_gap_mean_24h', 'anion_gap_max_24h', 'glucose_mean_24h', 'glucose_min_24h', 'glucose_max_24h', 'platelets_mean_24h', 'platelets_min_24h

,subject_id,hadm_id,icustay_id,gender,admission_type,insurance,ethnicity,dob,dod,intime,...,has_chf,has_diabetes,has_resp_failure,has_sepsis,has_anemia,has_cad,has_acidosis,n_diagnoses,n_prior_admissions,vasopressor_flag
0,10006,142345,206504,F,EMERGENCY,Medicare,BLACK/AFRICAN AMERICAN,2094-03-05,2165-08-12,2164-10-23 21:10:15,...,1,1,0,0,1,1,0,21,0,0
1,10011,105331,232110,F,EMERGENCY,Private,UNKNOWN/NOT SPECIFIED,2090-06-05,2126-08-28,2126-08-14 22:34:00,...,0,0,0,0,0,0,0,6,0,0
2,10013,165520,264446,F,EMERGENCY,Medicare,UNKNOWN/NOT SPECIFIED,2038-09-03,2125-10-07,2125-10-04 23:38:00,...,0,0,0,1,0,0,0,9,0,1
3,10017,199207,204881,F,EMERGENCY,Medicare,WHITE,2075-09-21,2152-09-12,2149-05-29 18:52:29,...,0,1,0,0,1,0,0,14,0,0
4,10019,177759,228977,M,EMERGENCY,Medicare,WHITE,2114-06-20,2163-05-15,2163-05-14 20:43:56,...,0,0,1,1,0,0,0,14,0,1
